# EV Charger Fault Model (TensorFlow)

This notebook trains a simple multi-class classifier for `detail_label` using the combined CSV.
Adjust `DATA_PATH` to point at the desired threshold variant (e.g., `.bak85`).

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf

# ---- Config ----
DATA_PATH = 'processed_data_combined.csv'  # change to .bak80/.bak85/.bak90 if needed
TARGET_COL = 'detail_label'
BATCH_SIZE = 4096
EPOCHS = 10
SEED = 42


In [ ]:
df = pd.read_csv(DATA_PATH)

# Convert numeric columns safely
for col in df.columns:
    if col not in ['sheet_name', 'begin_time', 'end_time']:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop non-feature columns
drop_cols = [
    'sheet_name', 'begin_time', 'end_time', 'transaction_id', 'id',
    'label', TARGET_COL
]
feature_cols = [c for c in df.columns if c not in drop_cols]

# Basic missing value handling
for c in feature_cols:
    df[c] = df[c].fillna(df[c].median())

print('rows:', len(df))
print('features:', feature_cols)
print('target distribution:', df[TARGET_COL].value_counts())


In [ ]:
# Train/val split
np.random.seed(SEED)
idx = np.random.permutation(len(df))
split = int(len(df) * 0.8)
train_idx, val_idx = idx[:split], idx[split:]

train_df = df.iloc[train_idx]
val_df = df.iloc[val_idx]

x_train = train_df[feature_cols].values.astype('float32')
y_train = train_df[TARGET_COL].values.astype('int64')
x_val = val_df[feature_cols].values.astype('float32')
y_val = val_df[TARGET_COL].values.astype('int64')

train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train)).shuffle(10000, seed=SEED).batch(BATCH_SIZE)
val_ds = tf.data.Dataset.from_tensor_slices((x_val, y_val)).batch(BATCH_SIZE)


In [ ]:
# Normalization layer
normalizer = tf.keras.layers.Normalization()
normalizer.adapt(x_train)

num_classes = int(df[TARGET_COL].nunique())

model = tf.keras.Sequential([
    tf.keras.Input(shape=(x_train.shape[1],)),
    normalizer,
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(num_classes, activation='softmax'),
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


In [ ]:
# Class weights for imbalance
counts = train_df[TARGET_COL].value_counts().to_dict()
max_count = max(counts.values())
class_weight = {k: max_count / v for k, v in counts.items()}
print('class_weight:', class_weight)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weight,
    verbose=1
)


In [ ]:
# Evaluation and confusion matrix
val_pred = model.predict(x_val, batch_size=BATCH_SIZE)
val_pred_cls = val_pred.argmax(axis=1)

cm = tf.math.confusion_matrix(y_val, val_pred_cls, num_classes=num_classes)
print('Confusion matrix:
', cm.numpy())


In [ ]:
# Save model
model.save('model_tf')
print('saved to model_tf/')
